##Gold Layer for Analysis of AF, SCB and UKÄ datasets

## Build `dim_occupation`

This dimension standardizes occupation data across JobTech and SCB by using SSYK as the common classification.

### Purpose
`dim_occupation` provides a conformed occupation dimension that can be shared across:
- `fact_job_postings`
- `fact_wages`
- `fact_employment`

### Source
The dimension is built from the occupation mapping table created during the harmonization step:
- `silver_jobtech.occupation_group_to_ssyk_mapping`

### Design choice
The dimension is SSYK-centered and preserves the original JobTech occupation group label.

In [0]:
CREATE SCHEMA IF NOT EXISTS gold_analysis

In [0]:
DESCRIBE silver_jobtech.occupation_group_to_ssyk_mapping;

## Create `dim_occupation`

This step creates the Gold occupation dimension from the occupation mapping table.

### Structure
The dimension includes:
- a surrogate key: `occupation_id`
- the standardized SSYK code and name
- the original JobTech occupation group label


In [0]:
CREATE OR REPLACE TABLE gold_analysis.dim_occupation AS
SELECT
  ROW_NUMBER() OVER (
    ORDER BY
      -- If occupation_code is null, replace it with 'ZZZZ'
      coalesce(occupation_code, 'ZZZZ'),
      occupation_group_label
  ) AS occupation_id,
  occupation_code AS ssyk_code,
  occupation_name AS ssyk_name,
  occupation_group_label AS jobtech_occupation_group_label
FROM silver_jobtech.occupation_group_to_ssyk_mapping;

In [0]:
SELECT *
FROM gold_analysis.dim_occupation
ORDER BY occupation_id
LIMIT 20;

## Validate `dim_occupation`

This step validates the Gold occupation dimension after creation.

### What is being checked
- total number of rows in the dimension
- consistency with the occupation mapping source

### Why this matters
This confirms that the conformed occupation dimension was created successfully and is ready to be used by Gold fact tables.

In [0]:
SELECT COUNT(*) AS dim_occupation_rows
FROM gold_analysis.dim_occupation;

In [0]:
SELECT COUNT(*) AS mapping_rows
FROM silver_jobtech.occupation_group_to_ssyk_mapping;

In [0]:


CREATE OR REPLACE TABLE gold_analysis.dim_date (
  date_id DATE,
  year INT,
  month INT,
  quarter INT,
  week INT,
  day_of_week STRING
);

Dim_date uses a natural key where date_id is the actual date. 

In [0]:
INSERT INTO gold_analysis.dim_date 
WITH date_range AS (
  SELECT
    explode(
      sequence(
        CAST((SELECT MIN(CAST(publication_date AS DATE)) FROM silver_jobtech.job_postings_silver_clean) AS DATE),
        date_add(CURRENT_DATE, 365),
        INTERVAL 1 DAY
      )
    ) AS full_date
),
date_details AS (
  SELECT
    full_date AS date_id,
    YEAR(full_date) AS year,
    MONTH(full_date) AS month,
    QUARTER(full_date) AS quarter,
    WEEKOFYEAR(full_date) AS week,
    DATE_FORMAT(full_date, 'EEEE') AS day_of_week
  FROM date_range
)
SELECT * FROM date_details;

In [0]:
CREATE OR REPLACE TABLE gold_analysis.dim_gender (
  gender_id INT,
  gender_name STRING,
  is_actual_gender BOOLEAN
);

In [0]:
CREATE OR REPLACE TABLE gold_analysis.dim_location (
  location_ID INT,
  municipality_name STRING,
  municipality_code STRING,
  region_name STRING
);

In [0]:
--ALTER TABLE gold_analysis.municipality_lan_lookup
--ADD COLUMN municipality_name VARCHAR(100);

UPDATE gold_analysis.municipality_lan_lookup
SET municipality_name = CASE municipality_code

-- STOCKHOLM COUNTY
WHEN '0114' THEN 'Upplands Väsby'
WHEN '0115' THEN 'Vallentuna'
WHEN '0117' THEN 'Österåker'
WHEN '0120' THEN 'Värmdö'
WHEN '0123' THEN 'Järfälla'
WHEN '0125' THEN 'Ekerö'
WHEN '0126' THEN 'Huddinge'
WHEN '0127' THEN 'Botkyrka'
WHEN '0128' THEN 'Salem'
WHEN '0138' THEN 'Tyresö'
WHEN '0139' THEN 'Upplands-Bro'
WHEN '0140' THEN 'Nykvarn'
WHEN '0160' THEN 'Täby'
WHEN '0162' THEN 'Danderyd'
WHEN '0163' THEN 'Sollentuna'
WHEN '0180' THEN 'Stockholm'
WHEN '0181' THEN 'Södertälje'
WHEN '0182' THEN 'Nacka'
WHEN '0183' THEN 'Sundbyberg'
WHEN '0184' THEN 'Solna'
WHEN '0186' THEN 'Lidingö'
WHEN '0187' THEN 'Vaxholm'
WHEN '0188' THEN 'Norrtälje'
WHEN '0191' THEN 'Sigtuna'
WHEN '0192' THEN 'Nynäshamn'

-- UPPSALA COUNTY
WHEN '0305' THEN 'Håbo'
WHEN '0319' THEN 'Älvkarleby'
WHEN '0330' THEN 'Knivsta'
WHEN '0360' THEN 'Tierp'
WHEN '0380' THEN 'Uppsala'
WHEN '0381' THEN 'Enköping'
WHEN '0382' THEN 'Östhammar'

-- SÖDERMANLAND
WHEN '0428' THEN 'Vingåker'
WHEN '0461' THEN 'Gnesta'
WHEN '0480' THEN 'Nyköping'
WHEN '0481' THEN 'Oxelösund'
WHEN '0482' THEN 'Flen'
WHEN '0483' THEN 'Katrineholm'
WHEN '0484' THEN 'Eskilstuna'
WHEN '0486' THEN 'Strängnäs'
WHEN '0488' THEN 'Trosa'

-- ÖSTERGÖTLAND
WHEN '0509' THEN 'Ödeshög'
WHEN '0512' THEN 'Ydre'
WHEN '0513' THEN 'Kinda'
WHEN '0560' THEN 'Boxholm'
WHEN '0561' THEN 'Åtvidaberg'
WHEN '0562' THEN 'Finspång'
WHEN '0563' THEN 'Valdemarsvik'
WHEN '0580' THEN 'Linköping'
WHEN '0581' THEN 'Norrköping'
WHEN '0582' THEN 'Söderköping'
WHEN '0583' THEN 'Motala'
WHEN '0584' THEN 'Vadstena'
WHEN '0586' THEN 'Mjölby'

-- JÖNKÖPING
WHEN '0604' THEN 'Aneby'
WHEN '0617' THEN 'Gnosjö'
WHEN '0642' THEN 'Mullsjö'
WHEN '0643' THEN 'Habo'
WHEN '0662' THEN 'Gislaved'
WHEN '0665' THEN 'Vaggeryd'
WHEN '0680' THEN 'Jönköping'
WHEN '0682' THEN 'Nässjö'
WHEN '0683' THEN 'Värnamo'
WHEN '0684' THEN 'Sävsjö'
WHEN '0685' THEN 'Vetlanda'
WHEN '0686' THEN 'Eksjö'
WHEN '0687' THEN 'Tranås'

-- KRONOBERG
WHEN '0760' THEN 'Uppvidinge'
WHEN '0761' THEN 'Lessebo'
WHEN '0763' THEN 'Tingsryd'
WHEN '0764' THEN 'Alvesta'
WHEN '0765' THEN 'Älmhult'
WHEN '0767' THEN 'Markaryd'
WHEN '0780' THEN 'Växjö'
WHEN '0781' THEN 'Ljungby'

-- KALMAR
WHEN '0821' THEN 'Högsby'
WHEN '0834' THEN 'Torsås'
WHEN '0840' THEN 'Mörbylånga'
WHEN '0860' THEN 'Hultsfred'
WHEN '0861' THEN 'Mönsterås'
WHEN '0862' THEN 'Emmaboda'
WHEN '0880' THEN 'Kalmar'
WHEN '0881' THEN 'Nybro'
WHEN '0882' THEN 'Oskarshamn'
WHEN '0883' THEN 'Västervik'
WHEN '0884' THEN 'Vimmerby'
WHEN '0885' THEN 'Borgholm'

-- GOTLAND
WHEN '0980' THEN 'Gotland'

-- BLEKINGE
WHEN '1060' THEN 'Olofström'
WHEN '1080' THEN 'Karlskrona'
WHEN '1081' THEN 'Ronneby'
WHEN '1082' THEN 'Karlshamn'
WHEN '1083' THEN 'Sölvesborg'

-- SKÅNE
WHEN '1230' THEN 'Staffanstorp'
WHEN '1231' THEN 'Burlöv'
WHEN '1233' THEN 'Vellinge'
WHEN '1256' THEN 'Östra Göinge'
WHEN '1257' THEN 'Örkelljunga'
WHEN '1260' THEN 'Bjuv'
WHEN '1261' THEN 'Kävlinge'
WHEN '1262' THEN 'Lomma'
WHEN '1263' THEN 'Svedala'
WHEN '1264' THEN 'Skurup'
WHEN '1265' THEN 'Sjöbo'
WHEN '1266' THEN 'Hörby'
WHEN '1267' THEN 'Höör'
WHEN '1270' THEN 'Tomelilla'
WHEN '1272' THEN 'Bromölla'
WHEN '1273' THEN 'Osby'
WHEN '1275' THEN 'Perstorp'
WHEN '1276' THEN 'Klippan'
WHEN '1277' THEN 'Åstorp'
WHEN '1278' THEN 'Båstad'
WHEN '1280' THEN 'Malmö'
WHEN '1281' THEN 'Lund'
WHEN '1282' THEN 'Landskrona'
WHEN '1283' THEN 'Helsingborg'
WHEN '1284' THEN 'Höganäs'
WHEN '1285' THEN 'Eslöv'
WHEN '1286' THEN 'Ystad'
WHEN '1287' THEN 'Trelleborg'
WHEN '1290' THEN 'Kristianstad'
WHEN '1291' THEN 'Simrishamn'
WHEN '1292' THEN 'Ängelholm'
WHEN '1293' THEN 'Hässleholm'

-- HALLAND
WHEN '1315' THEN 'Hylte'
WHEN '1380' THEN 'Halmstad'
WHEN '1381' THEN 'Laholm'
WHEN '1382' THEN 'Falkenberg'
WHEN '1383' THEN 'Varberg'
WHEN '1384' THEN 'Kungsbacka'

-- VÄSTRA GÖTALAND (partial due to length, but included fully logically)
WHEN '1401' THEN 'Härryda'
WHEN '1402' THEN 'Partille'
WHEN '1407' THEN 'Öckerö'
WHEN '1415' THEN 'Stenungsund'
WHEN '1419' THEN 'Tjörn'
WHEN '1421' THEN 'Orust'
WHEN '1427' THEN 'Sotenäs'
WHEN '1430' THEN 'Munkedal'
WHEN '1435' THEN 'Tanum'
WHEN '1438' THEN 'Dals-Ed'
WHEN '1439' THEN 'Färgelanda'
WHEN '1440' THEN 'Ale'
WHEN '1441' THEN 'Lerum'
WHEN '1442' THEN 'Vårgårda'
WHEN '1443' THEN 'Bollebygd'
WHEN '1444' THEN 'Grästorp'
WHEN '1445' THEN 'Essunga'
WHEN '1446' THEN 'Karlsborg'
WHEN '1447' THEN 'Gullspång'
WHEN '1452' THEN 'Tranemo'
WHEN '1460' THEN 'Bengtsfors'
WHEN '1461' THEN 'Mellerud'
WHEN '1462' THEN 'Lilla Edet'
WHEN '1463' THEN 'Mark'
WHEN '1465' THEN 'Svenljunga'
WHEN '1466' THEN 'Herrljunga'
WHEN '1470' THEN 'Vara'
WHEN '1471' THEN 'Götene'
WHEN '1472' THEN 'Tibro'
WHEN '1473' THEN 'Töreboda'
WHEN '1480' THEN 'Göteborg'
WHEN '1481' THEN 'Mölndal'
WHEN '1482' THEN 'Kungälv'
WHEN '1484' THEN 'Lysekil'
WHEN '1485' THEN 'Uddevalla'
WHEN '1486' THEN 'Strömstad'
WHEN '1487' THEN 'Vänersborg'
WHEN '1488' THEN 'Trollhättan'
WHEN '1489' THEN 'Alingsås'
WHEN '1490' THEN 'Borås'
WHEN '1491' THEN 'Ulricehamn'
WHEN '1492' THEN 'Åmål'
WHEN '1493' THEN 'Mariestad'
WHEN '1494' THEN 'Lidköping'
WHEN '1495' THEN 'Skara'
WHEN '1496' THEN 'Skövde'
WHEN '1497' THEN 'Hjo'
WHEN '1498' THEN 'Tidaholm'
WHEN '1499' THEN 'Falköping'

-- VÄRMLAND
WHEN '1715' THEN 'Kil'
WHEN '1730' THEN 'Eda'
WHEN '1737' THEN 'Torsby'
WHEN '1760' THEN 'Storfors'
WHEN '1761' THEN 'Hammarö'
WHEN '1762' THEN 'Munkfors'
WHEN '1763' THEN 'Forshaga'
WHEN '1764' THEN 'Grums'
WHEN '1765' THEN 'Årjäng'
WHEN '1766' THEN 'Sunne'
WHEN '1780' THEN 'Karlstad'
WHEN '1781' THEN 'Kristinehamn'
WHEN '1782' THEN 'Filipstad'
WHEN '1783' THEN 'Hagfors'
WHEN '1784' THEN 'Arvika'
WHEN '1785' THEN 'Säffle'

-- ÖREBRO
WHEN '1814' THEN 'Lekeberg'
WHEN '1860' THEN 'Laxå'
WHEN '1861' THEN 'Hallsberg'
WHEN '1862' THEN 'Degerfors'
WHEN '1863' THEN 'Hällefors'
WHEN '1864' THEN 'Ljusnarsberg'
WHEN '1880' THEN 'Örebro'
WHEN '1881' THEN 'Kumla'
WHEN '1882' THEN 'Askersund'
WHEN '1883' THEN 'Karlskoga'
WHEN '1884' THEN 'Nora'
WHEN '1885' THEN 'Lindesberg'

-- VÄSTMANLAND
WHEN '1904' THEN 'Skinnskatteberg'
WHEN '1907' THEN 'Surahammar'
WHEN '1960' THEN 'Kungsör'
WHEN '1961' THEN 'Hallstahammar'
WHEN '1962' THEN 'Norberg'
WHEN '1980' THEN 'Västerås'
WHEN '1981' THEN 'Sala'
WHEN '1982' THEN 'Fagersta'
WHEN '1983' THEN 'Köping'
WHEN '1984' THEN 'Arboga'

-- DALARNA
WHEN '2021' THEN 'Vansbro'
WHEN '2023' THEN 'Malung-Sälen'
WHEN '2026' THEN 'Gagnef'
WHEN '2029' THEN 'Leksand'
WHEN '2031' THEN 'Rättvik'
WHEN '2034' THEN 'Orsa'
WHEN '2039' THEN 'Älvdalen'
WHEN '2061' THEN 'Smedjebacken'
WHEN '2062' THEN 'Mora'
WHEN '2080' THEN 'Falun'
WHEN '2081' THEN 'Borlänge'
WHEN '2082' THEN 'Säter'
WHEN '2083' THEN 'Hedemora'
WHEN '2084' THEN 'Avesta'
WHEN '2085' THEN 'Ludvika'

-- GÄVLEBORG
WHEN '2101' THEN 'Ockelbo'
WHEN '2104' THEN 'Hofors'
WHEN '2121' THEN 'Ovanåker'
WHEN '2132' THEN 'Nordanstig'
WHEN '2161' THEN 'Ljusdal'
WHEN '2180' THEN 'Gävle'
WHEN '2181' THEN 'Sandviken'
WHEN '2182' THEN 'Söderhamn'
WHEN '2183' THEN 'Bollnäs'
WHEN '2184' THEN 'Hudiksvall'

-- VÄSTERNORRLAND
WHEN '2260' THEN 'Ånge'
WHEN '2262' THEN 'Timrå'
WHEN '2280' THEN 'Härnösand'
WHEN '2281' THEN 'Sundsvall'
WHEN '2282' THEN 'Kramfors'
WHEN '2283' THEN 'Sollefteå'
WHEN '2284' THEN 'Örnsköldsvik'

-- JÄMTLAND
WHEN '2303' THEN 'Ragunda'
WHEN '2305' THEN 'Bräcke'
WHEN '2309' THEN 'Krokom'
WHEN '2313' THEN 'Strömsund'
WHEN '2321' THEN 'Åre'
WHEN '2326' THEN 'Berg'
WHEN '2361' THEN 'Härjedalen'
WHEN '2380' THEN 'Östersund'

-- VÄSTERBOTTEN
WHEN '2401' THEN 'Nordmaling'
WHEN '2403' THEN 'Bjurholm'
WHEN '2404' THEN 'Vindeln'
WHEN '2409' THEN 'Robertsfors'
WHEN '2417' THEN 'Norsjö'
WHEN '2418' THEN 'Malå'
WHEN '2421' THEN 'Storuman'
WHEN '2422' THEN 'Sorsele'
WHEN '2425' THEN 'Dorotea'
WHEN '2460' THEN 'Vännäs'
WHEN '2462' THEN 'Vilhelmina'
WHEN '2463' THEN 'Åsele'
WHEN '2480' THEN 'Umeå'
WHEN '2481' THEN 'Lycksele'
WHEN '2482' THEN 'Skellefteå'

-- NORRBOTTEN
WHEN '2505' THEN 'Arvidsjaur'
WHEN '2506' THEN 'Arjeplog'
WHEN '2510' THEN 'Jokkmokk'
WHEN '2513' THEN 'Överkalix'
WHEN '2514' THEN 'Kalix'
WHEN '2518' THEN 'Övertorneå'
WHEN '2521' THEN 'Pajala'
WHEN '2523' THEN 'Gällivare'
WHEN '2560' THEN 'Älvsbyn'
WHEN '2580' THEN 'Luleå'
WHEN '2581' THEN 'Piteå'
WHEN '2582' THEN 'Boden'
WHEN '2583' THEN 'Haparanda'
WHEN '2584' THEN 'Kiruna'

-- SPECIAL
WHEN 'null' THEN NULL
WHEN 'ALLA' THEN 'All municipalities'

ELSE municipality_name
END;

In [0]:
TRUNCATE TABLE gold_analysis.dim_location;

INSERT INTO gold_analysis.dim_location
SELECT 
  ROW_NUMBER() OVER (ORDER BY region_name, municipality_code) AS location_id,
  municipality_name,
  municipality_code,
  region_name
FROM (
  -- Job ads: municipality_name + municipality_code + region
  SELECT DISTINCT 
    municipality.municipality_name,
    jobs.municipality_code, 
    municipality.lan_name AS region_name
  FROM silver_jobtech.job_postings_silver_clean AS jobs
  LEFT JOIN gold_analysis.municipality_lan_lookup AS municipality ON jobs.municipality_code = municipality.municipality_code
  WHERE jobs.municipality_code IS NOT NULL
  
  UNION
  
  -- Education UKÄ: region only
  SELECT NULL AS municipality_name, NULL AS municipality_code, region_name
  FROM silver_education.university_uka
  WHERE region_name IS NOT NULL
  
  UNION
  
  -- Education YH: region only
  SELECT NULL, NULL, region_name
  FROM silver_education.yh_region_subject
  WHERE region_name IS NOT NULL
  
  UNION
  
  -- Population: region only
  SELECT NULL, NULL, region
  FROM labour_market_platform.silver_work_scb.aku_population
  WHERE region IS NOT NULL
)
ORDER BY region_name, municipality_code;


In [0]:
SELECT * FROM gold_analysis.dim_location

In [0]:
TRUNCATE TABLE gold_analysis.dim_gender;

INSERT INTO gold_analysis.dim_gender
SELECT 
  CASE 
    WHEN gender_name = 'men' THEN 1
    WHEN gender_name = 'women' THEN 2
    WHEN LOWER(gender_name) = 'total' THEN 3
    ELSE 99
  END AS gender_id,
  gender_name,
  CASE 
    WHEN LOWER(gender_name) = 'total' THEN FALSE
    ELSE TRUE
  END AS is_actual_gender
FROM (
  SELECT DISTINCT gender AS gender_name FROM silver_education.university_uka
  UNION
  SELECT DISTINCT gender FROM silver_education.yh_region_subject
  UNION
  SELECT DISTINCT gender FROM labour_market_platform.silver_work_scb.aku_population
  UNION
  SELECT DISTINCT gender FROM labour_market_platform.silver_work_scb.aku_employment
  UNION
  SELECT DISTINCT gender FROM labour_market_platform.silver_work_scb.aku_unemployment
  UNION
  SELECT DISTINCT gender FROM labour_market_platform.silver_work_scb.wages
)
ORDER BY gender_id;

In [0]:
SELECT * FROM gold_analysis.dim_gender;

## Build `fact_job_postings`

### Grain
One row per job posting (`job_id`) from the cleaned and deduplicated JobTech Silver table.

### Planned dimensions
- `occupation_id`
- `date_id`
- `location_id`

### Planned measures / attributes
- `number_of_vacancies`
- `scope_of_work_min`
- `scope_of_work_max`
- `application_deadline`
- `employment_type_label`
- `duration_label`
- `working_hours_type_label`

### Source
`silver_jobtech.job_postings_silver_clean`

At this step, we first validate the join to `dim_occupation` before building the final fact table.

In [0]:
SELECT
  COUNT (*) AS total_job_postings,
  SUM(CASE WHEN d.occupation_id IS NOT NULL THEN 1 ELSE 0 END) AS matched_to_dim_occupation,
  SUM(CASE WHEN d.occupation_id IS NULL THEN 1 ELSE 0 END) AS unmatched_to_dim_occupation,
  ROUND(
        100.0 * SUM(CASE WHEN d.occupation_id IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS occupation_match_pct
FROM silver_jobtech.job_postings_silver_clean j
LEFT JOIN gold_analysis.dim_occupation d
ON j.occupation_group_label = d.jobtech_occupation_group_label;

## Occupation mapping validation

### Result
- Total job postings: 6,753,458
- Successfully mapped to dim_occupation: 99.82%

### Interpretation
The vast majority of job postings are successfully linked to SSYK occupation codes.

Unmatched rows (~0.18%) are likely due to:
- rare occupation labels
- edge cases in source data

### Conclusion
The mapping quality is sufficient for Gold layer modeling and analysis.

In [0]:
SELECT
    COUNT(*) AS total_job_postings,
    SUM(CASE WHEN l.location_ID IS NOT NULL THEN 1 ELSE 0 END) AS matched_to_dim_location,
    SUM(CASE WHEN l.location_ID IS NULL THEN 1 ELSE 0 END) AS unmatched_to_dim_location,
    ROUND(
        100.0 * SUM(CASE WHEN l.location_ID IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS location_match_pct
FROM silver_jobtech.job_postings_silver_clean j
LEFT JOIN gold_analysis.dim_location l
    ON j.municipality_code = l.municipality_code;

In [0]:
CREATE OR REPLACE TABLE gold_analysis.fact_job_postings AS
SELECT
    j.job_id,

    -- Foreign keys
    o.occupation_id,
    l.location_ID,
    d.date_id,

    -- Measures
    j.number_of_vacancies,
    j.scope_of_work_min,
    j.scope_of_work_max,

    -- Descriptive attributes
    j.application_deadline,
    j.employment_type_label,
    j.duration_label,
    j.working_hours_type_label

FROM silver_jobtech.job_postings_silver_clean j

LEFT JOIN gold_analysis.dim_occupation o
    ON j.occupation_group_label = o.jobtech_occupation_group_label

LEFT JOIN gold_analysis.dim_location l
    ON j.municipality_code = l.municipality_code

LEFT JOIN gold_analysis.dim_date d
    ON CAST(j.publication_date AS DATE) = d.date_id;

In [0]:
SELECT COUNT(*) AS fact_rows 
FROM gold_analysis.fact_job_postings;

In [0]:
SELECT * FROM gold_analysis.fact_job_postings LIMIT 20;

In [0]:
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN d.date_id IS NOT NULL THEN 1 ELSE 0 END) AS matched_dates,
    SUM(CASE WHEN d.date_id IS NULL THEN 1 ELSE 0 END) AS unmatched_dates,
    ROUND(100.0 * SUM(CASE WHEN d.date_id IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS match_pct
FROM silver_jobtech.job_postings_silver_clean j
LEFT JOIN gold_analysis.dim_date d
    ON CAST(j.publication_date AS DATE) = d.date_id;

## fact_job_postings completed

The final Gold fact table for job postings has been created.

### Foreign keys
- `occupation_id` → `dim_occupation`
- `location_ID` → `dim_location`
- `date_id` → `dim_date`

### Measures
- `number_of_vacancies`
- `scope_of_work_min`
- `scope_of_work_max`

### Descriptive attributes
- `application_deadline`
- `employment_type_label`
- `duration_label`
- `working_hours_type_label`

### Validation
The joins to occupation, location, and date dimensions were successfully validated, with near-complete coverage.

##Build fact_education



In [0]:
CREATE OR REPLACE TABLE gold_analysis.fact_education_graduates(
  date_id DATE,
  institution STRING,
  subject STRING,
  gender_id INT,
  location_id INT,
  student_count INT,
  program_not_offered BOOLEAN
);

In [0]:
INSERT INTO gold_analysis.fact_education_graduates
SELECT
  CAST(CONCAT(silver_education.university_uka.year, '-01-01') AS DATE) as date_id,
  silver_education
